# Linear Combination of Unitaries and Block Encoding Workbook

This workbook describes the solutions to the problems offered in the "Linear Combination of Unitaries and Block Encoding" kata. Since the tasks are offered as programming problems, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
from psiqdk.workbench import Qubits, Qubrick, units
from math import atan2, sqrt

### Problem 1.1. Prepare a single-qubit state

As we have seen in the earlier katas, a qubit in the $\ket{0}$ state can be converted to a superposition of states with real coefficients using an $Ry$ gate: 

$$R_y(\theta) = \begin{bmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{bmatrix}$$

This gate turns the state $\ket{0}$ into $Ry(\theta)\ket{0} = \cos\frac{\theta}{2} \ket{0} + \sin\frac{\theta}{2} \ket{1}$,
which is similar to the state you need. You just need to find an angle $\theta$ such that 

$$
\cos\frac{\theta}{2}=\sqrt{\frac{\alpha_0}{\alpha_0 + \alpha_1}} \\ 
\sin\frac{\theta}{2}=\sqrt{\frac{\alpha_1}{\alpha_0 + \alpha_1}}
$$

You can use these two equations to solve for $\theta$ to get $\theta = 2\arctan\sqrt\frac{\alpha_1}{\alpha_0}$.

Since we're guaranteed that $\alpha_k$ are positive, you don't need any special handling for negative or zero values: you can use the Python functions `atan2`, `acos`, or `asin` equally easily. Remember that these functions return angles in radians, and the $Ry$ gate takes the rotation angle in degrees. This means that you need to convert the angle to degrees, for example, by multiplying it by `units.rad`.

In [ ]:
class OneQubitPrepare(Qubrick):
    def _compute(self, reg: Qubits, alpha: list[float]) -> None:
        theta = 2 * atan2(sqrt(alpha[1]), sqrt(alpha[0]))
        reg.ry(theta * units.rad)

### Problem 1.2. Select between I and Z

This is a very simple example of a SELECT: we have only two unitaries, and one of them, $I$, does nothing. This means that the whole effect of the SELECT boils down to applying the $Z$ gate to the `target` register if the `index` register is in the $\ket{1}$ state - and that's simply a controlled $Z$ gate.

In [ ]:
class OneQubitSelectIZ(Qubrick):
    def _compute(self, index: Qubits, target: Qubits) -> None:
        target.z(cond=index)

### Problem 2.1. Find the LCU decomposition

If we focus only on the non-zero elements of the matrices involved, the LCU decomposition of the matrix $A$ looks as follows:

$$\begin{pmatrix}
    \beta_0 & & & \\
    & \beta_1 & & \\
    & & \beta_2 & \\
    & & & \beta_3 
\end{pmatrix} = \alpha_0 I \otimes I + \alpha_1 I \otimes Z + \alpha_2 Z \otimes I + \alpha_3 Z \otimes Z$$

$$
= \alpha_0 \begin{pmatrix}
    1 & & & \\
    & 1 & & \\
    & & 1 & \\
    & & & 1 
\end{pmatrix}
+ \alpha_1 \begin{pmatrix}
    1 & & & \\
    & -1 & & \\
    & & 1 & \\
    & & & -1 
\end{pmatrix}
+ \alpha_2 \begin{pmatrix}
    1 & & & \\
    & 1 & & \\
    & & -1 & \\
    & & & -1 
\end{pmatrix}
+ \alpha_3 \begin{pmatrix}
    1 & & & \\
    & -1 & & \\
    & & -1 & \\
    & & & 1 
\end{pmatrix}
$$

We can convert this matrix equality into a system of equations:

$$
\begin{cases}
\beta_0 = \alpha_0 + \alpha_1 + \alpha_2 + \alpha_3 \\
\beta_1 = \alpha_0 - \alpha_1 + \alpha_2 - \alpha_3 \\
\beta_2 = \alpha_0 + \alpha_1 - \alpha_2 - \alpha_3 \\
\beta_3 = \alpha_0 - \alpha_1 - \alpha_2 + \alpha_3
\end{cases}
$$

Solving this system, we get:

$$
\begin{cases}
\beta_0 + \beta_3 = 2 (\alpha_0 + \alpha_3) \\
\beta_0 - \beta_3 = 2 (\alpha_1 + \alpha_2) \\
\beta_1 + \beta_2 = 2 (\alpha_0 - \alpha_3) \\
\beta_2 - \beta_1 = 2 (\alpha_1 - \alpha_2) \\
\end{cases}
$$

$$
\begin{cases}
\alpha_0 = \tfrac14(\beta_0 + \beta_1 + \beta_2 + \beta_3) \\
\alpha_1 = \tfrac14(\beta_0 - \beta_1 + \beta_2 - \beta_3) \\
\alpha_2 = \tfrac14(\beta_0 + \beta_1 - \beta_2 - \beta_3) \\
\alpha_3 = \tfrac14(\beta_0 - \beta_1 - \beta_2 + \beta_3)
\end{cases}
$$

In [ ]:
def lcu_decomposition(beta: list[float]) -> list[float]:
    return [
        (beta[0] + beta[1] + beta[2] + beta[3]) / 4,
        (beta[0] - beta[1] + beta[2] - beta[3]) / 4,
        (beta[0] + beta[1] - beta[2] - beta[3]) / 4,
        (beta[0] - beta[1] - beta[2] + beta[3]) / 4
    ]

### Problem 2.2. Prepare a two-qubit state

This problem is exactly the problem 1.4 from the Preparing Arbitrary Quantum States kata; you can find its detailed solution and the steps leading to it in the [corresponding workbook](../ArbitraryStatePreparation/Workbook_ArbitraryStatePreparation.ipynb). The only difference is that here you need to implement the logic as a Qubrick rather than a plain function.

In [ ]:
class TwoQubitPrepare(Qubrick):
    def _compute(self, reg: Qubits, alpha: list[float]) -> None:
        a = [sqrt(a) for a in alpha]
        b0 = sqrt(a[0] ** 2 + a[1] ** 2)
        b1 = sqrt(a[2] ** 2 + a[3] ** 2)
        # Prepare the most significant qubit
        reg[1].ry(2 * atan2(b1, b0) * units.rad)
        # Prepare the least significant qubit for MSB=0
        reg[0].ry(2 * atan2(a[1], a[0]) * units.rad, cond=reg[1] == 0)
        # Prepare the least significant qubit for MSB=0
        reg[0].ry(2 * atan2(a[3], a[2]) * units.rad, cond=reg[1] == 1)

### Problem 2.3. Select between four I and Z tensor products

Once we split our registers into two separate qubits - the most and the least significant ones - we can reformulate the task as follows:

* If the least significant qubit of `index` is in the $\ket{1}$ state, apply the $Z$ gate to the least significant qubit of `target`, otherwise do nothing to it.
* If the most significant qubit of `index` is in the $\ket{1}$ state, apply the $Z$ gate to the most significant qubit of `target`, otherwise do nothing to it.

This is effectively the SELECT from problem 1.2, applied twice: once to the pair of most significant qubits of the respective register and once - to the pair of least significant qubits. We can write the same logic more concisely using the `cond_zip` argument of the `z` method; this "zips" the control and target registers together, applying a controlled gate between each pair of matching qubits.

In [ ]:
class TwoQubitSelectIZ(Qubrick):
    def _compute(self, index: Qubits, target: Qubits) -> None:
        target.z(cond_zip=index)

> Copyright (c) 2026 PsiQuantum